# Object Detection PASCAL VOC 2012 dan Evaluasi mAP

Dataset: [PASCAL VOC 2012 dari Kaggle](https://www.kaggle.com/datasets/gopalbhattrai/pascal-voc-2012-dataset).

Metode: **HOG + Linear SVM** bawaan OpenCV untuk mendeteksi objek kelas `person`. Evaluasi menghitung AP dan mAP@0.50.

In [ ]:
%pip install -q kagglehub opencv-python-headless numpy pandas matplotlib lxml

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CLASS_NAMES = ["person"]
print("OpenCV:", cv2.__version__)

## 1. Mengunduh data dari Kaggle

In [ ]:
DATASET_DIR = Path(kagglehub.dataset_download("gopalbhattrai/pascal-voc-2012-dataset"))
jpeg_dirs = sorted(DATASET_DIR.rglob("JPEGImages"))
dataset_roots = sorted({d.parent for d in jpeg_dirs})
paired_roots = [r for r in dataset_roots if (r / "Annotations").is_dir()
                and (r / "ImageSets" / "Main").is_dir()]
paired_roots = sorted(paired_roots, key=lambda r: ("train_val" not in str(r), str(r)))
if not paired_roots:
    raise FileNotFoundError("Struktur PASCAL VOC tidak ditemukan")

VOC_ROOT = paired_roots[0]
JPEG_DIR = VOC_ROOT / "JPEGImages"
ANNOTATION_DIR = VOC_ROOT / "Annotations"
MAIN_DIR = VOC_ROOT / "ImageSets" / "Main"

val_file = MAIN_DIR / "val.txt"
if val_file.exists():
    image_ids = [line.strip() for line in val_file.read_text().splitlines() if line.strip()]
else:
    image_ids = [p.stem for p in sorted(JPEG_DIR.glob("*.jpg"))]

EVAL_LIMIT = 100
image_ids = image_ids[:EVAL_LIMIT]
print("Lokasi dataset:", DATASET_DIR)
print("VOC root:", VOC_ROOT)
print("Jumlah citra evaluasi:", len(image_ids))

## 2. Model HOG + Linear SVM

In [ ]:
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

def detect_objects(image_bgr, confidence_threshold=0.50):
    boxes, weights = hog.detectMultiScale(
        image_bgr,
        winStride=(8, 8),
        padding=(8, 8),
        scale=1.05,
        finalThreshold=2
    )
    results = []
    for (x, y, w, h), weight in zip(boxes, weights):
        confidence = float(weight)
        if confidence < confidence_threshold:
            continue
        results.append({
            "class_name": "person",
            "confidence": confidence,
            "box": [int(x), int(y), int(x + w), int(y + h)]
        })
    return results

print("HOG/Linear SVM person detector berhasil dimuat")

## 3. Fungsi evaluasi AP/mAP

IoU minimum = 0.50. Ground truth `difficult=1` diabaikan. Satu ground truth hanya boleh dipasangkan satu kali.

In [ ]:
def parse_voc_annotation(annotation_path):
    root = ET.parse(annotation_path).getroot()
    objects = []
    for obj in root.findall("object"):
        name = obj.findtext("name")
        difficult = int(obj.findtext("difficult", "0"))
        box = obj.find("bndbox")
        if name != "person":
            continue
        objects.append({
            "class_name": name,
            "difficult": difficult,
            "box": [int(float(box.findtext("xmin"))) - 1,
                    int(float(box.findtext("ymin"))) - 1,
                    int(float(box.findtext("xmax"))) - 1,
                    int(float(box.findtext("ymax"))) - 1]
        })
    return objects

def box_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    intersection = max(0, x2 - x1 + 1) * max(0, y2 - y1 + 1)
    area1 = max(0, box1[2] - box1[0] + 1) * max(0, box1[3] - box1[1] + 1)
    area2 = max(0, box2[2] - box2[0] + 1) * max(0, box2[3] - box2[1] + 1)
    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0.0

def voc_ap(recall, precision):
    recall = np.concatenate(([0.0], recall, [1.0]))
    precision = np.concatenate(([0.0], precision, [0.0]))
    for i in range(len(precision) - 2, -1, -1):
        precision[i] = max(precision[i], precision[i + 1])
    indices = np.where(recall[1:] != recall[:-1])[0]
    return float(np.sum((recall[indices + 1] - recall[indices]) * precision[indices + 1]))

def compute_map(ground_truth, detections, iou_threshold=0.5):
    ap_per_class = {}
    for class_name in CLASS_NAMES:
        gt_items = {
            image_id: [obj["box"] for obj in ground_truth[image_id]
                       if obj["class_name"] == class_name and obj["difficult"] == 0]
            for image_id in ground_truth
        }
        gt_items = {image_id: boxes for image_id, boxes in gt_items.items() if boxes}
        if not gt_items:
            continue
        n_positive = sum(len(boxes) for boxes in gt_items.values())
        class_detections = []
        for image_id, image_detections in detections.items():
            for detection in image_detections:
                if detection["class_name"] == class_name:
                    class_detections.append((detection["confidence"], image_id, detection["box"]))
        class_detections.sort(key=lambda item: item[0], reverse=True)
        matched = {image_id: np.zeros(len(boxes), dtype=bool)
                   for image_id, boxes in gt_items.items()}
        tp = np.zeros(len(class_detections), dtype=np.float64)
        fp = np.zeros(len(class_detections), dtype=np.float64)
        for index, (_, image_id, predicted_box) in enumerate(class_detections):
            boxes = gt_items.get(image_id)
            if boxes is None:
                fp[index] = 1
                continue
            ious = [box_iou(predicted_box, gt_box) for gt_box in boxes]
            best_index = int(np.argmax(ious))
            if ious[best_index] >= iou_threshold and not matched[image_id][best_index]:
                tp[index] = 1
                matched[image_id][best_index] = True
            else:
                fp[index] = 1
        cumulative_tp = np.cumsum(tp)
        cumulative_fp = np.cumsum(fp)
        recall = cumulative_tp / n_positive
        precision = cumulative_tp / np.maximum(cumulative_tp + cumulative_fp,
                                               np.finfo(np.float64).eps)
        ap_per_class[class_name] = voc_ap(recall, precision)
    mean_ap = float(np.mean(list(ap_per_class.values()))) if ap_per_class else 0.0
    return mean_ap, ap_per_class

## 4. Menjalankan deteksi dan menghitung mAP

In [ ]:
ground_truth = {}
detections = {}
missing_files = 0

for image_id in image_ids:
    annotation_path = ANNOTATION_DIR / f"{image_id}.xml"
    image_path = JPEG_DIR / f"{image_id}.jpg"
    if not annotation_path.exists() or not image_path.exists():
        missing_files += 1
        continue
    ground_truth[image_id] = parse_voc_annotation(annotation_path)
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is not None:
        detections[image_id] = detect_objects(image_bgr, confidence_threshold=0.50)

mean_ap, ap_per_class = compute_map(ground_truth, detections, iou_threshold=0.5)
print("Citra valid:", len(ground_truth))
print("Citra tidak lengkap:", missing_files)
print(f"mAP@0.50: {mean_ap:.4f}")

In [ ]:
ap_df = pd.DataFrame([
    {"Kelas": class_name, "AP@0.50": ap_value}
    for class_name, ap_value in ap_per_class.items()
])
display(ap_df.style.format({"AP@0.50": "{:.4f}"}))

## 5. Contoh visualisasi

In [ ]:
sample_ids = list(detections.keys())[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, image_id in zip(axes.ravel(), sample_ids):
    image = cv2.imread(str(JPEG_DIR / f"{image_id}.jpg"))
    if image is None:
        ax.axis("off")
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    for detection in detections[image_id]:
        x1, y1, x2, y2 = detection["box"]
        cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(image, f'person: {detection["confidence"]:.2f}',
                    (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1)
    ax.imshow(image)
    ax.set_title(image_id)
    ax.axis("off")
plt.suptitle("Contoh hasil deteksi HOG/Linear SVM")
plt.tight_layout()
plt.show()